# Huấn luyện mô hình Gas-Leak (LSTM + PPO) trên Colab — **bản v2 đã sửa lỗi**

Notebook này huấn luyện lại hai mô hình của khóa luận **đã vá các lỗi** được chỉ ra trong phần đánh giá:

1. **LSTM Forecaster** — dự báo `P(gas > 1000 ppm trong 5 phút tới)`. Kiến trúc giữ nguyên (đã hợp lý).
2. **PPO RL Controller** — **environment đã sửa 3 lỗi**:
   - **(A) Reward hacking:** thưởng `+10` chỉ trao **đúng 1 lần** khi hành động *thực sự* kích hoạt can thiệp (không còn farm `+10`/bước bằng cách spam `FAN_ON` trong lúc rò rỉ).
   - **(B) Holding cost + auto-reset:** đóng van/bật quạt khi đang NORMAL bị phạt giữ-trạng-thái; khi phòng trở lại an toàn, van/quạt tự reset → mỗi sự kiện rò rỉ độc lập, agent phải phản ứng lại từng lần.
   - **(C) Reward chuẩn hoá:** dùng `VecNormalize` để đường cong reward diễn giải được (khắc phục việc reward kẹt ở vùng âm).
3. **Benchmark đã sửa** — thêm baseline **`rule`** (luật tay) để chứng minh PPO *học được gì hơn luật*; `miss` định nghĩa lại = *gas thực sự chạm 1000 ppm trong lượt điều khiển đó* (rõ ràng, công bằng cho mọi controller, không còn artifact lead-time 1803 s / miss 15%).

**Cách dùng:** `Runtime → Run all`. Cuối notebook sẽ tạo `artifacts_gas_v2.zip` chứa toàn bộ model + metrics + biểu đồ. Tải về và gửi lại vào Cowork để hoàn thiện LaTeX.

> Khuyến nghị: Runtime GPU không bắt buộc (mô hình nhỏ, CPU ~5–10 phút là xong).

> ⚠️ **Nếu gặp lỗi `numpy.dtype size changed`:** đó là do lần chạy trước đã hạ cấp numpy. Vào **Runtime → Restart session** rồi **Run all** lại — notebook này không còn hạ cấp numpy/TF nữa.


## 1. Cài đặt thư viện

In [ ]:
# ⚠️ QUAN TRỌNG: nếu trước đó bạn đã chạy bản notebook CŨ (pin numpy<2) và gặp lỗi
#    "numpy.dtype size changed" → vào menu Runtime → Restart session, rồi chạy lại từ ô này.
#
# Không hạ cấp numpy/tensorflow của Colab nữa (tránh vỡ ABI). Chỉ cài thêm RL libs.
!pip install -q "stable-baselines3>=2.4.0" kagglehub
import os, json, math, random, time
import numpy as np, tensorflow as tf
import gymnasium, stable_baselines3
os.makedirs("artifacts", exist_ok=True)
print("numpy", np.__version__, "| tensorflow", tf.__version__,
      "| gymnasium", gymnasium.__version__, "| sb3", stable_baselines3.__version__)

## 2. Bộ mô phỏng vật lý (state machine) — giữ nguyên như khóa luận

In [ ]:
%%writefile sensor_simulator.py
"""Realistic gas-sensor simulator (state machine). Identical physics to the thesis."""
import math, random
from dataclasses import dataclass
from enum import Enum

class State(str, Enum):
    NORMAL="NORMAL"; LEAK_SLOW="LEAK_SLOW"; LEAK_FAST="LEAK_FAST"; VENTILATING="VENTILATING"

@dataclass
class World:
    gas: float=60.0; temp: float=28.0; hum: float=60.0
    state: State=State.NORMAL; state_age_s: float=0.0

def _step_normal(w,dt):
    w.gas += (60.0-w.gas)*0.1*dt + random.gauss(0,3)*dt
    w.gas = max(20.0,min(w.gas,200.0))
    w.temp += random.gauss(0,0.05); w.hum += random.gauss(0,0.1)
def _step_leak_slow(w,dt):
    w.gas += 5.0*dt + random.gauss(0,2)*dt; w.gas=min(w.gas,1500.0); w.hum+=0.02*dt
def _step_leak_fast(w,dt):
    w.gas = w.gas*math.exp(0.04*dt) + 8.0*dt + random.gauss(0,3)*dt
    w.gas = min(w.gas,2000.0); w.hum+=0.05*dt
def _step_ventilating(w,dt):
    w.gas = max(60.0, w.gas*math.exp(-0.03*dt) - 0.5*dt)
    w.temp += random.gauss(0,0.05); w.hum -= 0.05*dt

STEP_FN={State.NORMAL:_step_normal, State.LEAK_SLOW:_step_leak_slow,
         State.LEAK_FAST:_step_leak_fast, State.VENTILATING:_step_ventilating}

import os
LEAK_PROB_PER_MIN=min(max(float(os.getenv("SIM_LEAK_PROB_PER_MIN","0.05")),0.0),1.0)

def _maybe_transition(w,dt):
    p=LEAK_PROB_PER_MIN/60.0*dt
    if w.state==State.NORMAL:
        if random.random()<p:
            w.state=State.LEAK_SLOW if random.random()<0.7 else State.LEAK_FAST
            w.state_age_s=0.0; return
    if w.state in (State.LEAK_SLOW,State.LEAK_FAST):
        if w.state_age_s>180 and random.random()<0.005:
            w.state=State.VENTILATING; w.state_age_s=0.0; return
    if w.state==State.VENTILATING:
        if w.gas<100 and w.state_age_s>30:
            w.state=State.NORMAL; w.state_age_s=0.0

def _seconds_to_critical(w):
    C=1000.0
    if w.gas>=C: return 0
    if w.state==State.LEAK_SLOW: return max(0,int((C-w.gas)/5.0))
    if w.state==State.LEAK_FAST:
        if w.gas<=0: return -1
        return max(0,int(math.log(C/w.gas)/0.04))
    return -1

## 3. Gymnasium Environment — **bản đã sửa lỗi**

So với bản gốc (`processing/ml/rl/gas_env.py`):

| Lỗi gốc | Sửa trong notebook này |
|---|---|
| `+10` trao mỗi bước khi `action∈{2,3}` trong leak (nằm ngoài guard) → **farm reward** | `+10` chỉ trao **trong** guard `if not fan_on/valve_closed` khi *thực sự* kích hoạt can thiệp |
| Van/quạt latch vĩnh viễn, không chi phí giữ | Phạt giữ-trạng-thái khi NORMAL; tự reset khi phòng an toàn |
| Reward kẹt vùng âm, khó diễn giải | Dùng `VecNormalize(norm_reward=True)` khi train |


In [ ]:
%%writefile gas_env.py
"""FIXED+v3 Gymnasium env for gas-leak response.
v3 changes vs v2: OUTCOME-BASED reward so the agent must pick the action that
actually keeps gas below CRITICAL. Fan alone cannot contain an exponential
fast leak (only CLOSE_VALVE can), so a per-step containment reward forces the
agent to learn decisive valve-closing on fast leaks instead of weak fanning.
"""
from __future__ import annotations
import math
from typing import Any, Optional
import numpy as np
try:
    import gymnasium as gym
    from gymnasium import spaces
except ImportError:
    import gym
    from gym import spaces
from sensor_simulator import State, World, STEP_FN, _maybe_transition, _seconds_to_critical

ACTION_NAMES=("NO_OP","ALERT_USER","FAN_ON","CLOSE_VALVE")
NUM_ACTIONS=4
CRITICAL_PPM=1000.0
HORIZON=300

class GasLeakEnv(gym.Env):
    metadata={"render_modes":[]}
    def __init__(self, episode_seconds:int=1800, seed:Optional[int]=None):
        super().__init__()
        self.episode_seconds=episode_seconds
        self._rng=np.random.default_rng(seed)
        self.observation_space=spaces.Box(low=0.0,high=1.0,shape=(8,),dtype=np.float32)
        self.action_space=spaces.Discrete(NUM_ACTIONS)
        self.world=World(); self.t=0
        self.fan_on=False; self.valve_closed=False
        self.time_since_action=0; self._gas_history=[]

    def reset(self,*,seed:Optional[int]=None,options:Any=None):
        if seed is not None: self._rng=np.random.default_rng(seed)
        self.world=World(); self.t=0
        self.fan_on=False; self.valve_closed=False
        self.time_since_action=0; self._gas_history=[]
        return self._observe(0.0), {}

    def step(self, action:int):
        cost=0.0
        leak_state=self.world.state
        leaking = leak_state in (State.LEAK_SLOW, State.LEAK_FAST)

        if action==1:                                   # ALERT_USER
            cost = -3.0 if leak_state==State.NORMAL else 0.0
            self.time_since_action=0
        elif action==2:                                 # FAN_ON
            if not self.fan_on:
                self.fan_on=True
                if leak_state==State.NORMAL: cost=-15.0
            self.time_since_action=0
        elif action==3:                                 # CLOSE_VALVE
            if not self.valve_closed:
                self.valve_closed=True
                if leaking:
                    self.world.state=State.VENTILATING; self.world.state_age_s=0.0
                else:
                    cost=-25.0
            self.time_since_action=0
        else:                                           # NO_OP
            self.time_since_action+=1

        # physics
        STEP_FN[self.world.state](self.world,1.0)
        if self.fan_on and self.world.state!=State.VENTILATING:
            self.world.gas=max(50.0,self.world.gas*0.985)
        _maybe_transition(self.world,1.0)
        self.world.state_age_s+=1.0; self.t+=1
        self._gas_history.append(self.world.gas)

        reward=-0.05+cost
        if self.world.gas>=CRITICAL_PPM:
            reward-=50.0                                # safety: heavy penalty at CRITICAL
        # v3 OUTCOME reward: while a leak is active, reward keeping gas contained.
        # Fan cannot contain a fast leak -> agent must CLOSE_VALVE to earn this.
        if leaking and self.world.gas<CRITICAL_PPM:
            reward+=0.3
        # holding cost while NORMAL (cut gas supply / wasted power)
        if self.world.state==State.NORMAL:
            if self.valve_closed: reward-=0.5
            if self.fan_on:       reward-=0.1
        # household restores valve/fan once room is safe again
        if self.world.state==State.NORMAL and self.world.state_age_s>20:
            self.valve_closed=False; self.fan_on=False

        terminated=False; truncated=self.t>=self.episode_seconds
        return self._observe(self._cheap_forecast()), float(reward), terminated, truncated, {
            "leak_state":self.world.state.value,"gas_ppm":self.world.gas,
            "ttc":_seconds_to_critical(self.world)}

    def _cheap_forecast(self):
        if len(self._gas_history)<5: return 0.0
        n=min(30,len(self._gas_history)); x=np.arange(n,dtype=np.float32)
        slope,_=np.polyfit(x,self._gas_history[-n:],1)
        predicted=self.world.gas+slope*HORIZON
        return float(1.0/(1.0+math.exp(-(predicted-CRITICAL_PPM)/200.0)))
    def _slope(self):
        if len(self._gas_history)<5: return 0.5
        n=min(30,len(self._gas_history)); x=np.arange(n,dtype=np.float32)
        slope,_=np.polyfit(x,self._gas_history[-n:],1)
        return float(np.clip((slope+10.0)/20.0,0.0,1.0))
    def _observe(self,p):
        return np.array([min(self.world.gas/2000.0,1.0),min(self.world.temp/60.0,1.0),
            min(self.world.hum/100.0,1.0),self._slope(),float(np.clip(p,0.0,1.0)),
            1.0 if self.fan_on else 0.0,1.0 if self.valve_closed else 0.0,
            min(self.time_since_action/300.0,1.0)],dtype=np.float32)

## 4. Huấn luyện LSTM Forecaster (dữ liệu simulator)
Sinh trace từ simulator, tạo cặp `(window 60s → P(critical trong 300s))`, split 80/20 theo thời gian (không shuffle, tránh leakage), class-weight chống mất cân bằng.

In [ ]:
import random, numpy as np, tensorflow as tf
from sensor_simulator import World, STEP_FN, _maybe_transition

SEQ_LEN, HORIZON, CRITICAL = 60, 300, 1000.0
HOURS = 8.0            # tăng lên 12-16 nếu muốn nhiều dữ liệu hơn
EPOCHS = 20
SEED = 42

def simulate(hours, seed=42):
    random.seed(seed); np.random.seed(seed)
    n=int(hours*3600); out=np.zeros((n,4),dtype=np.float32); w=World()
    s2i={"NORMAL":0,"LEAK_SLOW":1,"LEAK_FAST":2,"VENTILATING":3}
    for t in range(n):
        STEP_FN[w.state](w,1.0); _maybe_transition(w,1.0); w.state_age_s+=1.0
        out[t]=(w.gas,w.temp,w.hum,s2i[w.state.value])
    return out

def build_dataset(trace):
    feats=trace[:,:3]; gas=trace[:,0]; n=len(trace); X=[];y=[]
    for i in range(0, n-SEQ_LEN-HORIZON):
        X.append(feats[i:i+SEQ_LEN]/np.array([2000.0,60.0,100.0],dtype=np.float32))
        future=gas[i+SEQ_LEN:i+SEQ_LEN+HORIZON]
        y.append(1.0 if future.max()>CRITICAL else 0.0)
    return np.array(X,dtype=np.float32), np.array(y,dtype=np.float32)

trace=simulate(HOURS,SEED); X,y=build_dataset(trace)
split=int(len(X)*0.8)
Xtr,Xte,ytr,yte=X[:split],X[split:],y[:split],y[split:]
ppos=float(ytr.mean()); cw={0:1.0,1:(1-ppos)/max(ppos,1e-6)}
print(f"samples train={len(Xtr)} test={len(Xte)} pos_rate={ppos:.3f} class_weight_pos={cw[1]:.2f}")

inp=tf.keras.layers.Input(shape=(SEQ_LEN,3))
x=tf.keras.layers.LSTM(32,return_sequences=False)(inp)
x=tf.keras.layers.Dropout(0.2)(x)
x=tf.keras.layers.Dense(16,activation="relu")(x)
out=tf.keras.layers.Dense(1,activation="sigmoid")(x)
model=tf.keras.Model(inp,out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="binary_crossentropy",
              metrics=["accuracy",tf.keras.metrics.Precision(name="precision"),
                       tf.keras.metrics.Recall(name="recall")])
hist=model.fit(Xtr,ytr,validation_data=(Xte,yte),epochs=EPOCHS,batch_size=128,
               class_weight=cw,verbose=2)
model.save("artifacts/gas_forecaster.keras")

### 4b. Đánh giá LSTM + lưu metrics & biểu đồ

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

p=model.predict(Xte,verbose=0).ravel(); yhat=(p>0.5).astype(int)
tn,fp,fn,tp=confusion_matrix(yte,yhat,labels=[0,1]).ravel()
metrics={"samples_train":int(len(Xtr)),"samples_test":int(len(Xte)),
    "positive_rate":round(float(ppos),4),
    "accuracy":round(float(accuracy_score(yte,yhat)),4),
    "precision":round(float(precision_score(yte,yhat,zero_division=0)),4),
    "recall":round(float(recall_score(yte,yhat,zero_division=0)),4),
    "f1":round(float(f1_score(yte,yhat,zero_division=0)),4),
    "tp":int(tp),"fp":int(fp),"tn":int(tn),"fn":int(fn),
    "epochs":EPOCHS,"seq_len":SEQ_LEN,"horizon":HORIZON,"hours":HOURS,"seed":SEED}
json.dump(metrics,open("artifacts/lstm_metrics.json","w"),indent=2)
print(json.dumps(metrics,indent=2))

fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(hist.history["loss"],label="train"); ax[0].plot(hist.history["val_loss"],label="val")
ax[0].set_title("LSTM loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist.history["accuracy"],label="train"); ax[1].plot(hist.history["val_accuracy"],label="val")
ax[1].set_title("LSTM accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig("artifacts/lstm_training.png",dpi=130)
import csv
with open("artifacts/lstm_history.csv","w",newline="") as f:
    w=csv.writer(f); ks=list(hist.history.keys()); w.writerow(["epoch"]+ks)
    for e in range(EPOCHS): w.writerow([e+1]+[hist.history[k][e] for k in ks])
print("saved lstm metrics + plots")

## 4c. (Dữ liệu THẬT) Huấn luyện + đánh giá LSTM trên Kaggle Environmental Sensor 132K

Tái tạo phần kiểm chứng *sim-to-real* của khóa luận trên cảm biến thật (MQ-2/MQ-7/DHT22, 3 thiết bị Raspberry Pi).
Dataset: `garystafford/environmental-sensor-data-132k` → file `iot_telemetry_data.csv`.

**Cách lấy dữ liệu — chọn 1 trong 2:**
- **Cách A (tự động):** chạy ô dưới, nó dùng `kagglehub` tải public dataset (có thể hỏi đăng nhập Kaggle).
- **Cách B (thủ công):** tải `iot_telemetry_data.csv` từ Kaggle về máy → kéo thả vào panel **Files** của Colab (thư mục `/content`). Ô dưới sẽ tự nhận file nếu đã có.

> RL **không** chạy phần này — RL chỉ train trên simulator (xem giải thích ở mục 5). Phần Kaggle chỉ để kiểm chứng LSTM trên dữ liệu thật.

In [ ]:
# --- Lấy file CSV: ưu tiên file đã có trong /content, nếu chưa thì thử kagglehub ---
import os, glob, numpy as np, json
CSV=None
for cand in ["iot_telemetry_data.csv","/content/iot_telemetry_data.csv"]+glob.glob("/content/**/iot_telemetry_data.csv",recursive=True):
    if os.path.exists(cand): CSV=cand; break
if CSV is None:
    try:
        import kagglehub
        path=kagglehub.dataset_download("garystafford/environmental-sensor-data-132k")
        hits=glob.glob(os.path.join(path,"**","iot_telemetry_data.csv"),recursive=True)
        CSV=hits[0] if hits else None
    except Exception as e:
        print("kagglehub không tải được:",e)
if CSV is None:
    raise FileNotFoundError("Chưa có iot_telemetry_data.csv. Hãy kéo thả file vào panel Files (Cách B) rồi chạy lại ô này.")
print("Dùng CSV:",CSV)

import pandas as pd
df=pd.read_csv(CSV)
print(df.columns.tolist(), "rows:",len(df))

SEQ_LEN, HORIZON = 60, 300
FEATS=["lpg","temp","humidity"]
# Ngưỡng "nguy hiểm" = phân vị 95 của lpg trên toàn dataset (proxy cho leak, đúng như khóa luận)
LPG_THRESH=float(np.percentile(df["lpg"].values, 95))
print("LPG p95 threshold =", LPG_THRESH)

# Chuẩn hoá [0,1] theo biên dữ liệu
fmin=df[FEATS].min().values.astype(np.float32); fmax=df[FEATS].max().values.astype(np.float32)

def windows_for_device(d):
    d=d.sort_values("ts"); f=d[FEATS].values.astype(np.float32); lpg=d["lpg"].values.astype(np.float32)
    n=len(d); X=[]; y=[]
    norm=(f-fmin)/np.maximum(fmax-fmin,1e-9)
    for i in range(0, n-SEQ_LEN-HORIZON):
        X.append(norm[i:i+SEQ_LEN])
        fut=lpg[i+SEQ_LEN:i+SEQ_LEN+HORIZON]
        y.append(1.0 if fut.max()>LPG_THRESH else 0.0)
    return (np.array(X,dtype=np.float32), np.array(y,dtype=np.float32)) if X else (np.zeros((0,SEQ_LEN,3),np.float32),np.zeros((0,),np.float32))

# Chronological 80/20 TRÊN TỪNG THIẾT BỊ (tránh trộn thiết bị trong cùng cửa sổ)
Xtr_l,ytr_l,Xte_l,yte_l=[],[],[],[]
for dev,g in df.groupby("device"):
    Xd,yd=windows_for_device(g)
    if len(Xd)==0: continue
    s=int(len(Xd)*0.8)
    Xtr_l.append(Xd[:s]); ytr_l.append(yd[:s]); Xte_l.append(Xd[s:]); yte_l.append(yd[s:])
Xtr=np.concatenate(Xtr_l); ytr=np.concatenate(ytr_l)
Xte=np.concatenate(Xte_l); yte=np.concatenate(yte_l)
print(f"train={len(Xtr)} test={len(Xte)} pos_train={ytr.mean():.4f} pos_test={yte.mean():.4f}")

In [ ]:
# --- Train + đánh giá LSTM (cùng kiến trúc) trên dữ liệu thật, kèm 95% bootstrap CI ---
import tensorflow as tf
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

ppos=float(ytr.mean()); cw={0:1.0,1:(1-ppos)/max(ppos,1e-6)}
k_inp=tf.keras.layers.Input(shape=(SEQ_LEN,3))
kx=tf.keras.layers.LSTM(32)(k_inp); kx=tf.keras.layers.Dropout(0.2)(kx)
kx=tf.keras.layers.Dense(16,activation="relu")(kx)
k_out=tf.keras.layers.Dense(1,activation="sigmoid")(kx)
kaggle_model=tf.keras.Model(k_inp,k_out)
kaggle_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="binary_crossentropy",
                     metrics=["accuracy"])
kh=kaggle_model.fit(Xtr,ytr,validation_data=(Xte,yte),epochs=25,batch_size=256,
                    class_weight=cw,verbose=2)
kaggle_model.save("artifacts/gas_forecaster_kaggle.keras")

p=kaggle_model.predict(Xte,verbose=0).ravel(); yhat=(p>0.5).astype(int)
tn,fp,fn,tp=confusion_matrix(yte,yhat,labels=[0,1]).ravel()

# bootstrap 95% CI (2000 resamples)
rng=np.random.default_rng(42); accs=[];precs=[];recs=[];f1s=[]
idx=np.arange(len(yte))
for _ in range(2000):
    b=rng.choice(idx,len(idx),replace=True)
    yt=yte[b]; yp=yhat[b]
    accs.append(accuracy_score(yt,yp)); precs.append(precision_score(yt,yp,zero_division=0))
    recs.append(recall_score(yt,yp,zero_division=0)); f1s.append(f1_score(yt,yp,zero_division=0))
def ci(a): return [round(float(np.percentile(a,2.5)),4),round(float(np.percentile(a,97.5)),4)]
mk={"config":{"dataset":"garystafford/environmental-sensor-data-132k","csv":"iot_telemetry_data.csv",
        "seq_len":SEQ_LEN,"horizon":HORIZON,"epochs":25,"batch_size":256,
        "lpg_threshold_value":LPG_THRESH,"feature_mins":fmin.tolist(),"feature_maxs":fmax.tolist(),
        "n_train":int(len(Xtr)),"n_test":int(len(Xte)),
        "pos_frac_train":round(float(ytr.mean()),5),"pos_frac_test":round(float(yte.mean()),5)},
    "tp":int(tp),"fp":int(fp),"tn":int(tn),"fn":int(fn),
    "accuracy":round(float(accuracy_score(yte,yhat)),4),
    "precision":round(float(precision_score(yte,yhat,zero_division=0)),4),
    "recall":round(float(recall_score(yte,yhat,zero_division=0)),4),
    "f1":round(float(f1_score(yte,yhat,zero_division=0)),4),
    "ci_95":{"accuracy":ci(accs),"precision":ci(precs),"recall":ci(recs),"f1":ci(f1s)}}
json.dump(mk,open("artifacts/lstm_metrics_kaggle.json","w"),indent=2)
print(json.dumps({k:mk[k] for k in ["accuracy","precision","recall","f1","ci_95"]},indent=2))

fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(kh.history["loss"],label="train");ax[0].plot(kh.history["val_loss"],label="val")
ax[0].set_title("LSTM (Kaggle real) loss");ax[0].legend()
ax[1].plot(kh.history["accuracy"],label="train");ax[1].plot(kh.history["val_accuracy"],label="val")
ax[1].set_title("LSTM (Kaggle real) accuracy");ax[1].legend()
plt.tight_layout();plt.savefig("artifacts/lstm_training_kaggle.png",dpi=130)
print("saved Kaggle LSTM metrics + plot")

## 5. Huấn luyện PPO (env đã sửa) + `VecNormalize`
Reward được chuẩn hoá để đường cong hội tụ diễn giải được. Lưu cả `ppo_gas_agent.zip` và thống kê normalize.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.monitor import Monitor
from gas_env import GasLeakEnv

TOTAL_STEPS=400_000
venv=make_vec_env(lambda: GasLeakEnv(episode_seconds=1800), n_envs=4, seed=42,
                  monitor_dir="artifacts/ppo_monitor")
venv=VecNormalize(venv, norm_obs=False, norm_reward=True, clip_reward=10.0)
model_rl=PPO("MlpPolicy",venv,n_steps=512,batch_size=128,gae_lambda=0.95,gamma=0.99,
             learning_rate=3e-4,ent_coef=0.01,verbose=1,seed=42)
t0=time.time(); model_rl.learn(total_timesteps=TOTAL_STEPS)
print(f"PPO trained in {time.time()-t0:.0f}s")
model_rl.save("artifacts/ppo_gas_agent.zip")
venv.save("artifacts/vecnormalize.pkl")

### 5b. Đường cong reward (đọc từ Monitor logs)

In [ ]:
import glob, pandas as pd
import matplotlib.pyplot as plt
rows=[]
for fp in sorted(glob.glob("artifacts/ppo_monitor/*.monitor.csv")):
    d=pd.read_csv(fp,skiprows=1); d["cum_t"]=d["l"].cumsum(); rows.append(d)
mon=pd.concat(rows).sort_values("cum_t") if rows else None
if mon is not None and len(mon):
    mon["ma"]=mon["r"].rolling(50,min_periods=1).mean()
    plt.figure(figsize=(9,4))
    plt.plot(mon["cum_t"],mon["r"],alpha=0.25,label="episode reward")
    plt.plot(mon["cum_t"],mon["ma"],lw=2,label="moving avg (50)")
    plt.xlabel("timesteps"); plt.ylabel("episode reward"); plt.legend()
    plt.title("PPO learning curve (FIXED env + VecNormalize)")
    plt.tight_layout(); plt.savefig("artifacts/ppo_reward.png",dpi=130)
    mon[["cum_t","r","ma"]].to_csv("artifacts/ppo_progress.csv",index=False)
    print("final moving-avg reward:",round(float(mon['ma'].iloc[-1]),1))
else:
    print("no monitor logs found")

## 6. Benchmark đã sửa — 4 controller, định nghĩa `miss` rõ ràng
- **threshold**: cảnh báo khi gas > 800 ppm.
- **forecaster**: cảnh báo khi LSTM `p5 > 0.5`.
- **rule**: luật tay (rule-based fallback gốc) — *baseline để so với PPO*.
- **rl**: PPO đã huấn luyện.

`miss` = sự kiện rò rỉ mà gas **thực sự chạm 1000 ppm** trong lượt điều khiển đó (rõ ràng, công bằng, không phụ thuộc căn chỉnh oracle). `peak_gas`, `alarms/hr` đo trực tiếp. Đây là so sánh quan trọng nhất: **PPO có hơn luật tay không?**

In [ ]:
import random, numpy as np
from sensor_simulator import State, World, STEP_FN, _maybe_transition
from stable_baselines3 import PPO
CRITICAL=1000.0; ALARM_GAP=60

forecaster=tf.keras.models.load_model("artifacts/gas_forecaster.keras")
fc_hist={}
def predict_p5(gas,temp,hum):
    buf=fc_hist.setdefault("b",[]); buf.append((gas,temp,hum))
    if len(buf)<60: return 0.0
    arr=np.array(buf[-60:],dtype=np.float32)/np.array([2000.,60.,100.],dtype=np.float32)
    return float(forecaster.predict(arr[None,...],verbose=0).ravel()[0])

def rule_policy(gas,p5,slope,fan,valve):
    if gas>800 and not valve: return 3
    if p5>0.7 and not valve:  return 3
    if (p5>0.4 or slope>1.0) and not fan: return 2
    if p5>0.3: return 1
    return 0

rl=PPO.load("artifacts/ppo_gas_agent.zip")

def slope_of(buf):
    if len(buf)<5: return 0.0
    n=min(30,len(buf)); x=np.arange(n,dtype=np.float32)
    s,_=np.polyfit(x,np.array(buf[-n:],dtype=np.float32),1); return float(s)

def run_ctrl(ctrl, hours, seed):
    random.seed(seed); np.random.seed(seed); fc_hist.clear()
    n=int(hours*3600); w=World()
    in_leak=False; leak_peak=0.0; acted=False
    peaks=[]; leaks=0; reached_critical=0; misses=0
    alarms=0; last_alarm=-10**9; normal_s=0
    fan=False; valve=False; gbuf=[]; p5=0.0; tsa=0
    for t in range(n):
        STEP_FN[w.state](w,1.0)
        if fan and w.state!=State.VENTILATING: w.gas=max(50.0,w.gas*0.985)
        _maybe_transition(w,1.0); w.state_age_s+=1.0; gbuf.append(w.gas)
        leaking = w.state in (State.LEAK_SLOW,State.LEAK_FAST)
        if leaking and not in_leak:
            in_leak=True; leaks+=1; leak_peak=w.gas; acted=False; crit_hit=False
        elif in_leak:
            leak_peak=max(leak_peak,w.gas)
            if w.gas>=CRITICAL and not crit_hit: crit_hit=True
            if not leaking:
                peaks.append(leak_peak)
                if crit_hit: reached_critical+=1; misses+=1  # gas DID reach critical => miss
                in_leak=False; fan=False; valve=False
        else:
            normal_s+=1
        # decide
        if ctrl=="threshold":
            a=1 if w.gas>800 else 0
        elif ctrl=="forecaster":
            if t%5==0: p5=predict_p5(w.gas,w.temp,w.hum)
            a=1 if p5>0.5 else 0
        elif ctrl=="rule":
            if t%5==0: p5=predict_p5(w.gas,w.temp,w.hum)
            a=rule_policy(w.gas,p5,slope_of(gbuf),fan,valve)
        elif ctrl=="rl":
            if t%5==0: p5=predict_p5(w.gas,w.temp,w.hum)
            obs=np.array([min(w.gas/2000.,1.),min(w.temp/60.,1.),min(w.hum/100.,1.),
                float(np.clip((slope_of(gbuf)+10.)/20.,0,1)),float(np.clip(p5,0,1)),
                1.0 if fan else 0.0,1.0 if valve else 0.0,min(tsa/300.,1.)],dtype=np.float32)
            a=int(rl.predict(obs,deterministic=True)[0])
        if a==2: fan=True; tsa=0
        elif a==3:
            valve=True; tsa=0
            if in_leak and w.state in (State.LEAK_SLOW,State.LEAK_FAST):
                w.state=State.VENTILATING; w.state_age_s=0.0
        elif a==1: tsa=0
        else: tsa+=1
        if a!=0 and not in_leak:
            if (t-last_alarm)>ALARM_GAP: alarms+=1
            last_alarm=t
    return {"controller":ctrl,"leaks":leaks,"reached_critical":reached_critical,
            "miss_rate":round(misses/max(1,leaks),3),
            "alarms_per_hour":round(alarms/max(1e-9,normal_s/3600.0),2),
            "mean_peak_gas":round(float(np.mean(peaks)) if peaks else 0.0,1)}

SEEDS=[42,43,44]; HOURS_BM=4.0
agg={}
for ctrl in ["threshold","forecaster","rule","rl"]:
    runs=[run_ctrl(ctrl,HOURS_BM,s) for s in SEEDS]
    def ms(k): v=[r[k] for r in runs]; return round(float(np.mean(v)),2),round(float(np.std(v)),2)
    agg[ctrl]={"miss_rate":ms("miss_rate"),"alarms_per_hour":ms("alarms_per_hour"),
               "mean_peak_gas":ms("mean_peak_gas"),
               "leaks":int(np.mean([r["leaks"] for r in runs])),"runs":runs}
    print(f"{ctrl:11} miss%={agg[ctrl]['miss_rate']}  alarm/hr={agg[ctrl]['alarms_per_hour']}  peak_gas={agg[ctrl]['mean_peak_gas']}")
json.dump(agg,open("artifacts/benchmark_results.json","w"),indent=2)

## 7. Benchmark theo kịch bản (idle / slow / fast) — lead time hợp lệ
Mỗi kịch bản 1 leak xác định nên lead-time so với critical-time là *hợp lệ cho mọi controller*.

In [ ]:
def run_scenario(ctrl, scenario, seed, minutes=60):
    import os
    os.environ["SIM_LEAK_PROB_PER_MIN"]="0"
    import importlib, sensor_simulator as S; importlib.reload(S)
    from sensor_simulator import State, World, STEP_FN, _maybe_transition
    random.seed(seed); np.random.seed(seed); fc_hist.clear()
    n=minutes*60; w=World(); leak_t=600
    fan=False; valve=False; gbuf=[]; p5=0.0; tsa=0
    first_action=None; critical_t=None; peak=0.0; acted_in_leak=False
    # oracle critical time (no intervention) for this deterministic leak
    def oracle_crit():
        random.seed(seed); np.random.seed(seed)
        ww=World(); 
        for tt in range(n):
            if tt==leak_t: ww.state=(State.LEAK_SLOW if scenario=="slow" else State.LEAK_FAST); ww.state_age_s=0.0
            STEP_FN[ww.state](ww,1.0); ww.state_age_s+=1.0
            if ww.gas>=CRITICAL: return tt
        return None
    oc = oracle_crit() if scenario in ("slow","fast") else None
    random.seed(seed); np.random.seed(seed)
    for t in range(n):
        if scenario in ("slow","fast") and t==leak_t:
            w.state=(State.LEAK_SLOW if scenario=="slow" else State.LEAK_FAST); w.state_age_s=0.0
        STEP_FN[w.state](w,1.0)
        if fan and w.state!=State.VENTILATING: w.gas=max(50.0,w.gas*0.985)
        w.state_age_s+=1.0; gbuf.append(w.gas)
        leaking=w.state in (State.LEAK_SLOW,State.LEAK_FAST)
        if leaking: peak=max(peak,w.gas)
        if w.gas>=CRITICAL and critical_t is None: critical_t=t
        if ctrl=="threshold": a=1 if w.gas>800 else 0
        else:
            if t%5==0: p5=predict_p5(w.gas,w.temp,w.hum)
            if ctrl=="forecaster": a=1 if p5>0.5 else 0
            elif ctrl=="rule": a=rule_policy(w.gas,p5,slope_of(gbuf),fan,valve)
            else:
                obs=np.array([min(w.gas/2000.,1.),min(w.temp/60.,1.),min(w.hum/100.,1.),
                    float(np.clip((slope_of(gbuf)+10.)/20.,0,1)),float(np.clip(p5,0,1)),
                    1.0 if fan else 0.0,1.0 if valve else 0.0,min(tsa/300.,1.)],dtype=np.float32)
                a=int(rl.predict(obs,deterministic=True)[0])
        if leaking and a!=0 and first_action is None: first_action=t
        if a==2: fan=True; tsa=0
        elif a==3:
            valve=True; tsa=0
            if leaking: w.state=State.VENTILATING; w.state_age_s=0.0
        elif a==1: tsa=0
        else: tsa+=1
    lead = (oc-first_action) if (oc is not None and first_action is not None) else 0.0
    return {"lead_s":round(float(lead),1),"peak":round(float(peak),1),
            "reached_critical":bool(critical_t is not None)}

scen={}
for sc in ["slow","fast"]:
    for ctrl in ["threshold","forecaster","rule","rl"]:
        rs=[run_scenario(ctrl,sc,s) for s in [42,43,44]]
        scen[f"{sc}/{ctrl}"]={
            "lead_s":round(float(np.mean([r['lead_s'] for r in rs])),1),
            "peak":round(float(np.mean([r['peak'] for r in rs])),1),
            "reached_critical_rate":round(float(np.mean([r['reached_critical'] for r in rs])),2)}
        print(f"{sc:5}/{ctrl:11} lead={scen[f'{sc}/{ctrl}']['lead_s']:6}s  peak={scen[f'{sc}/{ctrl}']['peak']:7}  crit_rate={scen[f'{sc}/{ctrl}']['reached_critical_rate']}")
# restore default leak prob
import os; os.environ["SIM_LEAK_PROB_PER_MIN"]="0.05"
json.dump(scen,open("artifacts/scenarios_results.json","w"),indent=2)

## 8. Đo độ trễ inference (LSTM + RL)

In [ ]:
import time
def time_ms(fn,n=200):
    fn();  # warmup
    ts=[]
    for _ in range(n):
        t0=time.perf_counter(); fn(); ts.append((time.perf_counter()-t0)*1000)
    ts=np.array(ts); return {"mean":round(float(ts.mean()),3),"median":round(float(np.median(ts)),3),
        "p95":round(float(np.percentile(ts,95)),3),"p99":round(float(np.percentile(ts,99)),3),"n":n}
dummy=np.random.rand(1,60,3).astype(np.float32)
obs=np.zeros(8,dtype=np.float32)
lat={"forecaster_ms":time_ms(lambda: forecaster.predict(dummy,verbose=0)),
     "rl_choose_ms":time_ms(lambda: rl.predict(obs,deterministic=True))}
json.dump(lat,open("artifacts/latency_results.json","w"),indent=2)
print(json.dumps(lat,indent=2))

## 9. Đóng gói toàn bộ artifacts và tải về

In [ ]:
import shutil
shutil.make_archive("artifacts_gas_v2","zip","artifacts")
print("Files in artifacts/:")
for f in sorted(os.listdir("artifacts")): print("  ",f)
try:
    from google.colab import files
    files.download("artifacts_gas_v2.zip")
except Exception as e:
    print("Tải thủ công artifacts_gas_v2.zip từ panel Files bên trái.", e)

## 10. Gửi lại kết quả
Sau khi chạy xong, gửi lại **`artifacts_gas_v2.zip`** vào Cowork. Mình sẽ:
1. Đặt model vào `processing/ml/lstm/` và `processing/ml/rl/`.
2. Cập nhật các bảng/biểu đồ/khẳng định trong `thesis-latex/chapters/4-ThucNghiemDanhGia.tex` cho khớp số liệu mới (đặc biệt là so sánh **PPO vs rule** — chứng minh giá trị thật của tầng RL).
3. Sửa các câu trong Chương 1/3 đang phát biểu quá mạnh về "đóng góp khoa học" của RL nếu số liệu chưa ủng hộ.
